# 02 FinBERT Sentiment

`chunks.parquet` を読み込み、FinBERT (`yiyanghkust/finbert-tone`) で
各チャンクの positive / negative / neutral 確率を推論する。
filing × section 単位で集約して可視化。

FinBERT は **金融テキスト特化の BERT-base** を `Sequence Classification`
(3 ラベル) で fine-tune したモデル。入力上限は **512 トークン**、出力は
`{Neutral, Positive, Negative}` の確率分布 (softmax)。

In [ ]:
# Cell 1: imports + setup
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import _helpers
import torch
import pandas as pd
from tqdm.auto import tqdm

device = _helpers.get_device()
print('device:', device)


## Cell 2: FinBERT モデルとトークナイザを直接ロード

`AutoTokenizer.from_pretrained` で tokenizer、
`AutoModelForSequenceClassification.from_pretrained` で分類モデル本体を
ロードする (Hugging Face Hub から、HF_HOME に既にキャッシュされていれば
そこから読む)。

**`AutoModelForSequenceClassification`** は文・段落単位を 1 ベクトル化
([CLS] トークンの埋め込み) して N クラス分類するためのヘッド付きモデル
クラス。NER (token-level) や embedding 専用とは別物。

ロード後は:
- `.to(device)` で GPU/MPS に移動
- `.eval()` で **学習用 dropout を無効化** (推論時は必須)
- `model.config.id2label` で出力ラベル名を確認

In [ ]:
# Cell 2: FinBERT (yiyanghkust/finbert-tone) を直接ロード
from transformers import AutoModelForSequenceClassification, AutoTokenizer

FINBERT_MODEL_ID = 'yiyanghkust/finbert-tone'

tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL_ID)
model.to(device)
model.eval()  # 推論モード (dropout を無効化)

print('model:', type(model).__name__)
print('num_labels:', model.config.num_labels)
print('id2label:', model.config.id2label)
print('hidden_size:', model.config.hidden_size)
print('max position embeddings:', model.config.max_position_embeddings)


In [ ]:
# Cell 3: chunks 読み込み (01 で保存した chunks.parquet を入力に使う)
df_chunks = pd.read_parquet(_helpers.CHUNKS_PARQUET)
print('chunks:', len(df_chunks))
df_chunks.head(3)


## Cell 4: FinBERT 推論ループ (バッチ処理)

全 chunk テキストを `BATCH_SIZE=32` 件ずつまとめて推論する。

**ステップ**:
1. `tokenizer(batch, padding=True, truncation=True, max_length=512,
   return_tensors='pt')` でテキスト → tensor。`padding=True` でバッチ内
   最長に合わせ、`truncation=True` で 512 を超える分を切る。
2. `.to(device)` で MPS / CPU に転送。
3. `torch.no_grad()` で **勾配計算を無効** にして高速化 + メモリ節約。
4. `model(**enc)` で `logits` (生スコア、形状 [B, 3]) を得る。
5. `.softmax(dim=-1)` で確率分布 (合計 1.0) に変換。
6. `.cpu().numpy()` で NumPy 配列に持ってくる。

**id2label の順序** (Neutral=0 / Positive=1 / Negative=2) はモデルに依存
するため、固定 index で扱わず Cell 5 で `id2label` を介して pos/neg/neu に
再マッピングする。

In [ ]:
# Cell 4: バッチ推論 (pos/neg/neu の確率を all_probs に蓄積)
BATCH_SIZE = 32
id2label = model.config.id2label  # {0: 'Neutral', 1: 'Positive', 2: 'Negative'}
label_names = [id2label[i] for i in range(len(id2label))]

all_probs = []
texts = df_chunks['text'].tolist()
for start in tqdm(range(0, len(texts), BATCH_SIZE), desc='finbert'):
    batch = texts[start:start+BATCH_SIZE]
    enc = tokenizer(
        batch,
        padding=True, truncation=True, max_length=512,
        return_tensors='pt',
    ).to(device)
    with torch.no_grad():
        out = model(**enc)
    probs = out.logits.softmax(dim=-1).cpu().numpy()
    all_probs.extend(probs.tolist())
print('done. samples:', len(all_probs), 'label_names:', label_names)


In [ ]:
# Cell 5: sentiments.parquet 保存 (id2label の順序に依らず pos/neg/neu 列で揃える)
import numpy as np
probs_arr = np.array(all_probs)
# ラベル名 → 列 index の写像を作り、明示的に pos/neg/neu の 3 列に再配置
name2idx = {n.lower(): i for i, n in enumerate(label_names)}
df_sent = df_chunks[['filing_id','ticker','form','section_key','chunk_idx','filing_date']].copy()
df_sent['pos'] = probs_arr[:, name2idx['positive']]
df_sent['neg'] = probs_arr[:, name2idx['negative']]
df_sent['neu'] = probs_arr[:, name2idx['neutral']]
df_sent['label'] = [label_names[i] for i in probs_arr.argmax(axis=1)]
df_sent.to_parquet(_helpers.SENTIMENTS_PARQUET)
print('saved:', _helpers.SENTIMENTS_PARQUET, 'rows:', len(df_sent))
df_sent.head()


In [ ]:
# Cell 6: filing × section 単位で平均センチメント集約
agg = df_sent.groupby(['ticker','form','section_key','filing_id','filing_date'])[['pos','neg','neu']].mean().reset_index()
agg = agg.sort_values(['ticker','section_key','filing_date'])
agg.head(10)


In [ ]:
# Cell 7: AAPL の Risk Factors (Item 1A) の neg スコア推移
import plotly.express as px
aapl_risk = agg[(agg['ticker']=='AAPL') & (agg['section_key']=='item_1a')].copy()
fig = px.line(
    aapl_risk, x='filing_date', y='neg', color='form', markers=True,
    title='AAPL Risk Factors (Item 1A) - FinBERT negative score over time',
)
fig.show()


In [ ]:
# Cell 8: 3 銘柄 × MD&A センチメント比較 (pos - neg)
mda = agg[agg['section_key']=='item_7'].copy()
mda['net_sentiment'] = mda['pos'] - mda['neg']
fig = px.line(
    mda, x='filing_date', y='net_sentiment', color='ticker', markers=True, line_dash='form',
    title='MD&A net sentiment (pos - neg) by ticker/form',
)
fig.show()
